# 🚀 ComfyUI + MiniMax-H3 on Google Colab (A100 GPU Edition)

Google AI Pro 等のプランで付与される **Colab Compute Units (CU)** を活用して、強力な最新動画生成モデル **MiniMax-H3 (Hailuo)** を A100 (40GB VRAM) 環境で動かすためのオールインワン検証テンプレートです。

> 📖 **詳細な解説・検証記事 (Zenn)**: [Google AI Pro(2,900円)に課金するとA100 40GBがColab経由で毎月37時間分使える話](https://zenn.dev/grand2/articles/ddba80ba400f6f)

### 💡 特徴・ポイント
- **A100 GPU 最適化**: MiniMax-H3 の大規模モデル (`int8_convrot` 等) + Text Encoder + Audio/Video VAE をストレスなくロード。
- **🚀 Turbo LoRA 最適化 (推奨)**: 通常 25 ステップ（約30分）かかる生成を、**わずか 6〜8 ステップ（約7分・所要時間 1/4）へ激変** させる Turbo LoRA を自動配備。
- **高速化スタック対応**: `SageAttention`、`TeaCache` などの最新高速化ノード群をトグル1つで自動セットアップ。
- **Google Drive 永続化 & ローカル SSD 最適化**:
  - モデルは Colab のローカル SSD 上に配置し、Drive (FUSE) 経由の読み込みによる WebUI の低速化・フリーズを回避。
  - モデル・LoRA（約20GB）は `MyDrive/ComfyUI_Models/` にキャッシュし、2回目以降は HF からの DL をスキップして Drive → ローカル SSD へ並列コピー。コピーは Step 1 からバックグラウンドで進み、インストールや ComfyUI の起動と並行するため A100 の待ち時間を最小化。
  - 生成された動画・画像は **`MyDrive/ComfyUI_Outputs/` に自動保存**。ランタイムが切断・終了しても成果物が消えません。
- **Cloudflare Tunnel (無料・トークン不要)**: ngrok 不要ですぐにセキュアな一時公開 URL (`trycloudflare.com`) を自動発行。

> ⚠️ **注意**: ノートブック上部のメニュー「ランタイム」→「ランタイムのタイプを変更」から、**GPU (A100)** が選択されていることを確認してください。

## Step 1: 環境確認 & Google Drive マウント & モデル準備の開始

- **⚡ モデル準備はバックグラウンドで実行**: このセルはすぐ終わり、モデル (約20GB) の準備は裏で進みます。Step 2 のインストールや Step 4 の ComfyUI 起動と並行するので、A100 を待たせる時間を最小限にできます。
- **⚡ モデルはローカル SSD に配置**: Drive 上のモデルを直接読むと WebUI の起動・操作が大幅に遅くなるため、モデルの実体は常に `/content/ComfyUI/models/` に置きます。
- **🚀 Turbo LoRA (`DOWNLOAD_TURBO_LORA = True`)**: `lightx2v/Minimax-h3-Turbo` をダウンロードし、**8ステップサンプリング（生成時間1/4）** を可能にします。
- `USE_GOOGLE_DRIVE = True` の場合：
  - **モデルキャッシュ**: `MyDrive/ComfyUI_Models/` にキャッシュがあればローカルへ並列コピー（Drive は 1 ストリームだと遅いため分割して同時に読み込み）、なければ Hugging Face から DL し、準備完了後に Drive にも保存します（次回以降 DL スキップ）。
  - **動画出力先**: `MyDrive/ComfyUI_Outputs/` (インスタンス終了後も成果物を保持)
  - **入力素材**: `MyDrive/ComfyUI_Inputs/` の素材を起動時にローカルへコピーします。**WebUI からアップロードした素材は Drive に保存されない** ので、残したい素材は Drive 側のフォルダに置いてください。

In [ ]:
# GPU および CUDA バージョンの確認 (A100 がアサインされているか確認)
!nvidia-smi

import os
import subprocess
import threading
import time
from concurrent.futures import ThreadPoolExecutor

# バックグラウンドでの DL 中に進捗バーが他のセルへ混ざらないようにする
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
from huggingface_hub import hf_hub_download

# @markdown **USE_GOOGLE_DRIVE**: ON にすると Google Drive をマウントし、以下を行います（OFF の場合はすべてランタイム内のみで完結し、切断時に消えます）。
# @markdown - モデルを `MyDrive/ComfyUI_Models/` にキャッシュ（2回目以降は DL 不要）
# @markdown - 生成物を `MyDrive/ComfyUI_Outputs/` に自動保存
# @markdown - `MyDrive/ComfyUI_Inputs/` の素材を起動時に読み込み
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
# @markdown **DOWNLOAD_TURBO_LORA**: 8ステップ生成用の Turbo LoRA をダウンロードします（推奨）。
DOWNLOAD_TURBO_LORA = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")

COMFY_DIR = "/content/ComfyUI"
LOCAL_MODELS_DIR = os.path.join(COMFY_DIR, "models")
LOCAL_OUTPUT_DIR = os.path.join(COMFY_DIR, "output")
LOCAL_INPUT_DIR = os.path.join(COMFY_DIR, "input")

DRIVE_BASE = "/content/drive/MyDrive"
DRIVE_MODELS_DIR = os.path.join(DRIVE_BASE, "ComfyUI_Models")
DRIVE_OUTPUT_DIR = os.path.join(DRIVE_BASE, "ComfyUI_Outputs")
DRIVE_INPUT_DIR = os.path.join(DRIVE_BASE, "ComfyUI_Inputs")

MIN_MODEL_SIZE = 1_000_000
COPY_WORKERS = 8          # Drive からの並列読み込み数
COPY_BLOCK = 64 << 20     # 並列コピーの分割単位 (64MB)

def rsync(src, dst):
    !rsync -a -h --info=progress2 "{src}" "{dst}"
    if _exit_code != 0:
        raise RuntimeError(f"rsync に失敗しました (exit {_exit_code}): {src} -> {dst}")

def is_valid_model(path):
    return os.path.isfile(path) and os.path.getsize(path) > MIN_MODEL_SIZE

def ensure_local_dir(path):
    # 旧バージョンで作られた Drive へのシンボリックリンクを外し、ローカルの実ディレクトリにする
    if os.path.islink(path):
        os.unlink(path)
    os.makedirs(path, exist_ok=True)

def parallel_copy(src, dst, progress, workers=COPY_WORKERS):
    # Drive (FUSE) は 1 ストリームあたりの読み込みが遅いため、ファイルをブロックに分割して並列に読む
    size = os.path.getsize(src)
    tmp = dst + ".part"
    blocks = [(off, min(COPY_BLOCK, size - off)) for off in range(0, size, COPY_BLOCK)]
    lock = threading.Lock()
    progress.update(copied=0, total=size, copy_start=time.time())

    with open(tmp, "wb") as f:
        f.truncate(size)

    def copy_block(block):
        off, length = block
        src_fd = os.open(src, os.O_RDONLY)
        dst_fd = os.open(tmp, os.O_WRONLY)
        try:
            done = 0
            while done < length:
                buf = os.pread(src_fd, length - done, off + done)
                if not buf:
                    raise IOError(f"予期せぬ EOF: {src} (offset {off + done})")
                view = memoryview(buf)
                while view:
                    written = os.pwrite(dst_fd, view, off + done)
                    view = view[written:]
                    done += written
        finally:
            os.close(src_fd)
            os.close(dst_fd)
        with lock:
            progress["copied"] += length

    try:
        with ThreadPoolExecutor(workers) as ex:
            list(ex.map(copy_block, blocks))
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise RuntimeError(f"コピー後のサイズが一致しません: {src}")
    os.replace(tmp, dst)

# 1. ComfyUI 本体を先にクローン (モデルの配置先を用意するため。依存関係のインストールは Step 2)
if not os.path.exists(COMFY_DIR):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}

# 2. 出力先: Google Drive にシンボリックリンク (生成物を即時永続化)
if USE_GOOGLE_DRIVE:
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    if os.path.exists(LOCAL_OUTPUT_DIR) and not os.path.islink(LOCAL_OUTPUT_DIR):
        !rm -rf {LOCAL_OUTPUT_DIR}
    if not os.path.exists(LOCAL_OUTPUT_DIR):
        os.symlink(DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR)
    print(f"📁 出力先を Google Drive に連携: {DRIVE_OUTPUT_DIR}")
else:
    ensure_local_dir(LOCAL_OUTPUT_DIR)

# 3. 入力素材: Google Drive からローカル SSD へコピー (リンクすると WebUI の一覧表示が遅くなるため)
ensure_local_dir(LOCAL_INPUT_DIR)
if USE_GOOGLE_DRIVE:
    os.makedirs(DRIVE_INPUT_DIR, exist_ok=True)
    print(f"🖼️ 入力素材を Google Drive からコピー中: {DRIVE_INPUT_DIR}")
    rsync(DRIVE_INPUT_DIR + "/", LOCAL_INPUT_DIR + "/")

# 4. モデル定義
REPO_ID = "Comfy-Org/MiniMax-H3"
files_to_download = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", REPO_ID),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", REPO_ID),
    ("vae/minimax_h3_video_vae_fp16.safetensors", REPO_ID),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", REPO_ID)
]

if DOWNLOAD_TURBO_LORA:
    files_to_download.append((
        "loras/minimax_h3_fl2v_turbo_8step_v1.0_768p_bf16.safetensors",
        "lightx2v/Minimax-h3-Turbo"
    ))

# 5. モデル準備 (ローカル → Drive キャッシュ → Hugging Face の順に探す) をバックグラウンドで実行
def prepare_models(state):
    try:
        for sub in sorted({os.path.dirname(f) for f, _ in files_to_download}):
            ensure_local_dir(os.path.join(LOCAL_MODELS_DIR, sub))

        to_cache = []
        for filename, repo in files_to_download:
            local_path = os.path.join(LOCAL_MODELS_DIR, filename)
            drive_path = os.path.join(DRIVE_MODELS_DIR, filename)
            sub_folder = os.path.dirname(filename)
            file_name_only = os.path.basename(filename)
            state.update(copied=0, total=0)

            if is_valid_model(local_path):
                state["step"] = f"ローカルSSDに配置済み: {filename}"
            elif USE_GOOGLE_DRIVE and is_valid_model(drive_path):
                state["step"] = f"Drive キャッシュから並列コピー中 ({COPY_WORKERS}並列): {filename}"
                parallel_copy(drive_path, local_path, state)
            else:
                state["step"] = f"Hugging Face からダウンロード中: {filename} (from {repo})"
                hf_hub_download(
                    repo_id=repo,
                    filename=file_name_only if repo != REPO_ID else filename,
                    local_dir=os.path.join(LOCAL_MODELS_DIR, sub_folder) if repo != REPO_ID else LOCAL_MODELS_DIR,
                )

            if not is_valid_model(local_path):
                raise RuntimeError(f"モデルの配置に失敗しました: {local_path}")
            if USE_GOOGLE_DRIVE and not is_valid_model(drive_path):
                to_cache.append((local_path, drive_path))

        state.update(done=True, elapsed=time.time() - state["started"], cache_total=len(to_cache))
    except Exception as e:
        state["error"] = f"{type(e).__name__}: {e}"
        return

    # 次回起動の高速化のため Drive にキャッシュ (生成を待たせないようモデル準備完了後に実行)
    try:
        for local_path, drive_path in to_cache:
            state["caching"] = os.path.relpath(drive_path, DRIVE_MODELS_DIR)
            os.makedirs(os.path.dirname(drive_path), exist_ok=True)
            subprocess.run(["rsync", "-a", local_path, drive_path], check=True, capture_output=True)
        state["cache_done"] = True
    except Exception as e:
        state["cache_error"] = f"{type(e).__name__}: {e}"
    finally:
        state["caching"] = None

def model_prep_status():
    s = MODEL_PREP
    if s["error"]:
        return f"❌ モデル準備に失敗しました: {s['error']}\n   Step 1 を再実行すると、配置済みのモデルはスキップして再開します。"
    if s["done"]:
        msg = f"✅ モデル準備完了 ({s['elapsed']:.0f} 秒)"
        if s["cache_error"]:
            msg += f"\n⚠️ Drive へのキャッシュ保存に失敗しました (次回は Hugging Face から再DL): {s['cache_error']}"
        elif s["caching"]:
            msg += f"\n💾 Drive へキャッシュ保存中: {s['caching']} (完了前に切断すると次回は Hugging Face から再DL)"
        return msg
    msg = f"⏳ モデル準備中 ({time.time() - s['started']:.0f} 秒経過) {s['step']}"
    if s["total"]:
        rate = s["copied"] / 1e6 / max(time.time() - s["copy_start"], 1e-6)
        msg += f"  {s['copied'] / 1e9:.1f} / {s['total'] / 1e9:.1f} GB ({rate:.0f} MB/s)"
    return msg

if "model_prep_thread" in globals() and model_prep_thread.is_alive():
    print("ℹ️ モデル準備はすでにバックグラウンドで実行中です。")
else:
    MODEL_PREP = dict(started=time.time(), step="開始待ち", copied=0, total=0, copy_start=0.0,
                      done=False, error=None, elapsed=0.0,
                      caching=None, cache_total=0, cache_done=False, cache_error=None)
    model_prep_thread = threading.Thread(target=prepare_models, args=(MODEL_PREP,), daemon=True)
    model_prep_thread.start()
    print("🚀 モデル準備をバックグラウンドで開始しました。完了を待たずに次のセルへ進んでください。")

## Step 2: ComfyUI 本体 & 高速化スタックのセットアップ

Step 1 で開始したモデル準備と並行して実行されます。

- **ENABLE_ACCELERATION (推奨)**: `SageAttention` / `Triton` をインストールします。
- **INSTALL_TEACACHE (推奨)**: `TeaCache` 高速化ノードをインストールします。
- **CUDA 12.8 / 13.0 互換性ハンドリング**: PyTorch cu128 とホスト CUDA 13.0 の互換レイヤーを安全に処理します。

In [ ]:
import os

# @title 高速化オプション設定
# @markdown **ENABLE_ACCELERATION**: SageAttention / Triton をインストールします（推奨）。
ENABLE_ACCELERATION = True  # @param {type:"boolean"}
# @markdown **INSTALL_TEACACHE**: TeaCache 高速化ノードをインストールします（推奨）。
INSTALL_TEACACHE = True     # @param {type:"boolean"}

%cd /content

# ComfyUI 本体のクローン (最新版)
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    %cd /content/ComfyUI
    !git pull
    %cd /content

# 依存ライブラリのインストール
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# SageAttention & Triton セットアップ
if ENABLE_ACCELERATION:
    print("⚡ 高速化スタック (SageAttention, Triton 等) をセットアップ中...")
    !pip install -q triton sageattention

# ComfyUI-Manager のインストール
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

# MiniMax-H3 Easy ノード
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy"):
    !git clone https://github.com/kijai/ComfyUI-MiniMaxH3-Easy.git || true

# TeaCache 高速化ノード
if INSTALL_TEACACHE and not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TeaCache"):
    !git clone https://github.com/chengzeyi/ComfyUI-MiniMaxH3-TeaCache.git || true

%cd /content/ComfyUI
print("✅ ノードおよび依存ライブラリの準備が完了しました！")
print(model_prep_status())

## Step 3: モデル準備の進捗確認

Step 1 から裏で進めているモデル準備の状況を表示します（**このセルは完了を待ちません**）。
Step 4 はモデル準備の完了を待たずに ComfyUI を起動し、完了するとそのセルの出力に通知が出ます。

In [ ]:
print(model_prep_status())

## Step 4: ComfyUI 起動 & Cloudflare Tunnel 経由でアクセス

バックグラウンドで ComfyUI を起動し、Cloudflare Tunnel (`trycloudflare.com`) の安全なパブリック URL を発行して**起動状態を維持**します。
表示された `https://xxxx.trycloudflare.com` のリンクをクリックすると ComfyUI の WebUI が開きます。

> ⏳ モデル準備が終わる前でも ComfyUI は起動します。このセルに **「✅ モデル準備完了」** と表示されたら、WebUI で **`R` キー**（ノード定義の更新）を押すとモデルが選択肢に表示されます。

### 💡 Turbo LoRA を使った超高速ワークフローの組み方
1. ComfyUI を開いたら、モデルローダー（`UNETLoader`）の直後に **`LoraLoader`** を挟みます。
2. LoRA に **`minimax_h3_fl2v_turbo_8step_...safetensors`** を選択します。
3. **KSampler の steps を `25` → `8` に変更** します（Euler / normal 推奨）。
4. これで **約 6〜8 分で 1 本の美麗動画が完成** します！

In [ ]:
import re
import subprocess
import threading
import time

# Cloudflared のダウンロード
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# ComfyUI をバックグラウンドで起動 (モデル準備の完了は待たない)
print("⚡ Starting ComfyUI...")
print(model_prep_status())
%cd /content/ComfyUI
comfy_proc = subprocess.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--highvram"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# ComfyUI の起動ログを別スレッドで監視
def log_comfy():
    for line in iter(comfy_proc.stdout.readline, ''):
        print("[ComfyUI]", line, end="")

threading.Thread(target=log_comfy, daemon=True).start()

# 少し待機して Cloudflared トンネルを起動
time.sleep(5)
print("🌐 Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8188"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None

def watch_tunnel():
    global tunnel_url
    for line in iter(tunnel_proc.stdout.readline, ''):
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match and tunnel_url is None:
            tunnel_url = match.group(0)

threading.Thread(target=watch_tunnel, daemon=True).start()

# URL の表示とモデル準備の進捗通知を行いつつ、トンネルプロセスが生きている間はセルを終了させず維持
url_shown = False
models_reported = False
cache_reported = False
last_progress = time.time()
try:
    while tunnel_proc.poll() is None:
        if tunnel_url and not url_shown:
            print("\n" + "="*60)
            print(f"🎉 ComfyUI is LIVE: {tunnel_url}")
            print("="*60 + "\n")
            print("💡 サーバーを稼働維持しています。(終了したい場合は本セルの停止ボタンを押すか、上部メニューから切断してください)")
            url_shown = True

        if not models_reported:
            if MODEL_PREP["done"] or MODEL_PREP["error"]:
                print(model_prep_status())
                if MODEL_PREP["done"]:
                    print("💡 WebUI で R キーを押すとモデル一覧が更新されます。")
                models_reported = True
            elif time.time() - last_progress >= 60:
                print(model_prep_status())
                last_progress = time.time()
        elif not cache_reported and MODEL_PREP["cache_total"] and (MODEL_PREP["cache_done"] or MODEL_PREP["cache_error"]):
            if MODEL_PREP["cache_error"]:
                print(f"⚠️ Drive へのキャッシュ保存に失敗しました: {MODEL_PREP['cache_error']}")
            else:
                print("💾 Drive へのモデルキャッシュ保存が完了しました (次回はダウンロード不要)")
            cache_reported = True

        time.sleep(2)
except KeyboardInterrupt:
    print("\n⏹️ サーバーを停止しました。")
    comfy_proc.terminate()
    tunnel_proc.terminate()

## 🛑 (作業完了時) 確実に課金をストップする方法

動画生成の検証が完了したら、**最も確実に Compute Units (CU) の消費をストップ** するため、以下の手順でインスタンスを破棄してください：

### 💡 【推奨・確実度 100%】Colab メニューから切断
1. Colab 画面上部メニューの **「ランタイム」** をクリック
2. **「ランタイムの接続を解除して削除」** を選択（確認画面で「はい」）

> `USE_GOOGLE_DRIVE = True` の場合、成果物（動画）は Google Drive（`MyDrive/ComfyUI_Outputs/`）にリアルタイム保存されているため、切断しても安全です。
> `False` にしている場合、または WebUI からアップロードした入力素材はランタイム削除とともに消えるので、必要なものは事前にダウンロードしてください。